# 03｜MuJoCo 模型、状态与控制

对应[第 03 章](../course/03-mujoco-model-and-control.md)。先用一个只有 1 个关节的模型看清数据，再回到 Panda；不要一开始就在完整 XML 中找答案。

## 学习目标

区分 MJCF、`MjModel`、`MjData`；读取 `qpos/qvel/ctrl/xpos`；解释控制步与物理步；把最小模型概念迁移到 Panda。

## 本节知识地图

| 知识点 | 一句话解释 | 掌握检查 |
| --- | --- | --- |
| MJCF | 机器人和场景的 XML 源描述 | 能找到 joint/geom/actuator |
| `MjModel` | 编译后的固定结构和参数 | 能读 `nq/nv/nu` |
| `MjData` | 一次运行的可变状态和派生量 | 能区分 `qpos` 与 `xpos` |
| actuator control | `ctrl` 是执行器输入，不一定等于关节位置 | 能预测 position actuator 行为 |
| timestep | 多个物理步组成一个策略控制步 | 算出 `ctrl_dt/sim_dt` |

## 关键概念与符号

`nq` 是位置坐标数，`nv` 是速度自由度数，`nu` 是 actuator 数；三者不保证相等。`xpos` 是前向计算得到的世界坐标，不是可直接写入的关节状态。

## 开始前诊断

先回答：① XML 属于 Model 还是 Data？② 给 `ctrl=0.2` 后 `qpos` 是否瞬间变成 0.2？③ `sim_dt=0.005`、`ctrl_dt=0.05` 含多少物理步？

## 先预测

模型只有一个 x 方向 slide joint 和一个 position actuator。预测 `nq/nv/nu`，再预测控制目标 0.2 m 后前 5 个物理步的 `qpos` 是逐渐变化还是一步到位。

## 运行与观察

第一段加载项目 kernel 和 MuJoCo。所有实验在 Mac CPU 上数秒内完成。

In [ ]:
from pathlib import Path
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(ROOT / 'docs' / 'notebooks'))

import ipywidgets as widgets
import matplotlib.pyplot as plt
import mujoco
import numpy as np
from IPython.display import display
from course_feedback import check_choice, check_value, save_progress
from course_utils import assert_course_kernel

assert_course_kernel(ROOT)
print('MuJoCo:', mujoco.__version__)

## Worked example

下面的 XML 依次声明 world、可滑动 body、joint、geom 和 position actuator。编译 XML 得到 `MjModel`，再创建初始为零的 `MjData`。

In [ ]:
MINIMAL_XML = '''
<mujoco model="course_slider">
  <option timestep="0.005" gravity="0 0 0"/>
  <worldbody>
    <body name="slider" pos="0 0 0.1">
      <joint name="slide_x" type="slide" axis="1 0 0" range="-0.3 0.3"/>
      <geom name="box" type="box" size="0.04 0.04 0.04" mass="1"/>
      <site name="tip" pos="0.04 0 0"/>
    </body>
  </worldbody>
  <actuator><position name="slide_target" joint="slide_x" kp="40"/></actuator>
</mujoco>
'''
model = mujoco.MjModel.from_xml_string(MINIMAL_XML)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
print(f'nq={model.nq}, nv={model.nv}, nu={model.nu}')
print('qpos:', data.qpos.copy(), 'ctrl:', data.ctrl.copy(), 'body xpos:', data.body('slider').xpos.copy())

In [ ]:
mujoco.mj_resetData(model, data)
data.ctrl[0] = 0.2
history = []
for step in range(80):
    mujoco.mj_step(model, data)
    mujoco.mj_forward(model, data)  # recompute xpos for the newly integrated qpos
    history.append((data.time, data.qpos[0], data.qvel[0], data.body('slider').xpos[0]))
history = np.asarray(history)
print('first five qpos:', np.round(history[:5, 1], 6))
print('final qpos:', round(float(history[-1, 1]), 4))
plt.figure(figsize=(7, 3.5))
plt.plot(history[:, 0], history[:, 1], label='qpos')
plt.plot(history[:, 0], history[:, 3], '--', label='body xpos')
plt.axhline(0.2, color='0.3', linestyle=':', label='ctrl target')
plt.xlabel('simulation time (s)'); plt.ylabel('x (m)'); plt.legend(); plt.show()

## 故意出错

错误说法：‘`ctrl` 是关节位置，所以写入后 `qpos` 立即相等。’用第一个物理步的数值反驳它。另一个常见错误是把 `xpos` 当成可写状态；它实际由 `qpos` 前向计算。

In [ ]:
first_step_relation = 'instant'  # 故意错误：改成 dynamic
check_choice(
    'actuator semantics', first_step_relation, 'dynamic',
    hint=f'first qpos={history[0, 1]:.6f}, target=0.2',
    explanation='ctrl is a target processed by actuator dynamics and integration.',
)

## 动手修改

移动滑块后重新运行下一格。先预测目标符号改变时曲线方向；不要改 XML。

In [ ]:
target_slider = widgets.FloatSlider(value=0.15, min=-0.25, max=0.25, step=0.05, description='target x')
display(target_slider)

In [ ]:
trial = mujoco.MjData(model)
trial.ctrl[0] = target_slider.value
for _ in range(100):
    mujoco.mj_step(model, trial)
mujoco.mj_forward(model, trial)
print(f'target={target_slider.value:+.2f}, qpos={trial.qpos[0]:+.4f}, xpos={trial.body("slider").xpos[0]:+.4f}')

### 控制步与物理步

Panda 的 `sim_dt=0.005 s`，策略的 `ctrl_dt=0.05 s`。同一动作在 10 个物理积分步内保持。改变 `sim_dt` 会改变积分分辨率；改变 `ctrl_dt` 会改变策略发命令的频率。

In [ ]:
sim_dt = 0.005
ctrl_dt = 0.05
n_substeps = round(ctrl_dt / sim_dt)
check_value('Panda physics substeps', n_substeps, 10, hint='divide control period by simulation timestep')

## 自测

运行前分别解释每个维度和状态量。

In [ ]:
assert (model.nq, model.nv, model.nu) == (1, 1, 1)
assert history[0, 1] != 0.2
assert np.allclose(history[:, 1], history[:, 3])
assert n_substeps == 10
print('PASS: model/data, actuator dynamics, derived xpos, and substeps')

## 项目源码连接

在 Panda 中，XML 仍编译成 MuJoCo model，运行状态仍在 data；但 MJX 将数组放到设备并批处理。下一步查看 [`mjx_single_cube_camera.xml`](../../mujoco_playground/_src/manipulation/franka_emika_panda/xmls/mjx_single_cube_camera.xml) 和 [`pick_cartesian.py`](../../mujoco_playground/_src/manipulation/franka_emika_panda/pick_cartesian.py)。

## Exit ticket

① `qpos` 与 `xpos` 谁是状态、谁是派生量？② position actuator 的 `ctrl` 是什么？③ 为什么一个控制步含 10 个物理步？

In [ ]:
exit_answers = {'xpos': 'derived', 'ctrl': 'target', 'substeps': 10}
exit_ticket_passed = all([
    check_choice('xpos role', exit_answers['xpos'], 'derived', hint='mj_forward updates it.', explanation='xpos is derived world position.'),
    check_choice('ctrl role', exit_answers['ctrl'], 'target', hint='actuator applies dynamics.', explanation='position actuator ctrl is a target.'),
    check_value('substeps', exit_answers['substeps'], 10, hint='0.05 / 0.005'),
])
SAVE_PROGRESS = False
if SAVE_PROGRESS:
    save_progress(ROOT, '03-mujoco', {'model-data': 'green', 'control': 'green', 'timestep': 'green'}, exit_ticket_passed=exit_ticket_passed)

## 学完请记住

1. MJCF 是源描述；2. Model 保存结构，Data 保存运行状态；3. `ctrl` 经过执行器和积分才改变 `qpos`；4. `xpos` 由前向计算得到；5. Panda 一个控制步含 10 个物理步。

## 反思与记录

在 `notes/03-mujoco-state.md` 画出 XML → Model → Data → ctrl → `mj_step` → `xpos`，然后完成第 03 章源码任务。